In [3]:
from pathlib import Path 
import numpy as np 
import pandas as pd 
from sklearn.compose import ColumnTransformer
from sklearn.metrics import(f1_score, precision_score, recall_score, roc_auc_score, accuracy_score, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder 
from sklearn.base import clone 
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

In [4]:
def load_data(data_dir: Path):
    X_train = pd.read_csv(data_dir / "X_train.csv")
    y_train = pd.read_csv(data_dir / "y_train.csv")

    # Flatten y if it is a column vector
    y_train = y_train.values.ravel()

    # Encode string labels to integers
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)

    return X_train, y_train


def load_test_data(data_dir: Path):
    X_test = pd.read_csv(data_dir / "X_test.csv")
    y_test = pd.read_csv(data_dir / "y_test.csv")

    # Flatten y if it is a column vector
    y_test = y_test.values.ravel()

    # Encode string labels to integers
    le = LabelEncoder()
    y_test = le.fit_transform(y_test)

    return X_test, y_test


def build_pipeline(X: pd.DataFrame):
    # Determine numeric vs categorical columns
    numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
    print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

    numeric_transformer = Pipeline(
        steps=[
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )

    clf = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ]
    )

    return pipe


def plot_roc_curves(best_model, X_train, y_train, X_test, y_test, output_dir: Path):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    y_prob_train= cross_val_predict(
        best_model,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba"
    )[:, 1]
    y_prob_test = best_model.predict_proba(X_test)[:, 1]

    for split, y_true, y_prob in [
        ("train", y_train, y_prob_train),
        ("test",  y_test,  y_prob_test),
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc:.4f}")
        ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random")
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel("False Positive Rate", fontsize=13)
        ax.set_ylabel("True Positive Rate", fontsize=13)
        ax.set_title(f"ROC Curve — {split.capitalize()} Set", fontsize=14)
        ax.legend(loc="lower right", fontsize=12)
        ax.grid(alpha=0.3)

        out_path = output_dir / f"xgboost_roc_{split}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved ROC curve ({split}) to {out_path}")


def main():
    data_dir = Path("data")
    X_train, y_train = load_data(data_dir)

    pipe = build_pipeline(X_train)

    # StratifiedKFold for stable splits
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    param_grid = {
        "classifier__n_estimators": [100],
        "classifier__max_depth": [3, 5],
        "classifier__learning_rate": [0.01],
        "classifier__subsample": [1.0],
        "classifier__colsample_bytree": [1.0]
    }

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        n_jobs=-1,
        cv=cv,
        verbose=1,
        refit=True,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_
    print("\nBest performing XGBoost model:")
    print(best_model)
    print("\nBest hyperparameters:")
    print(grid.best_params_)
    print("\nBest CV AUC:", f"{grid.best_score_:.4f}")

    # ROC curves
    X_test, y_test = load_test_data(data_dir)
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    plot_roc_curves(best_model, X_train, y_train, X_test, y_test, results_dir)

    scoring = {
        "accuracy": make_scorer(accuracy_score),
        "precision": make_scorer(precision_score, pos_label=1),
        "recall": make_scorer(recall_score, pos_label=1),
        "f1": make_scorer(f1_score, pos_label=1),
        "auc": "roc_auc",
    }
  
    scores_best = cross_validate(
        best_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    accuracy_best = scores_best["test_accuracy"].mean()
    precision_best = scores_best["test_precision"].mean()
    recall_best = scores_best["test_recall"].mean()
    f1_best = scores_best["test_f1"].mean()
    auc_best = scores_best["test_auc"].mean()

    print(("Error metrics for best performing XGBoost model on training data (averaged across folds):"))
    print(f"Accuracy:              {accuracy_best:.4f}")
    print(f"Precision (Malignant): {precision_best:.4f}")
    print(f"Recall (Malignant):    {recall_best:.4f}")
    print(f"F1 (Malignant):        {f1_best:.4f}")
    print(f"AUC:                   {auc_best:.4f}")

    # Metrics for test set
    y_pred_test = best_model.predict(X_test)
    y_prob_test = best_model.predict_proba(X_test)[:, 1]

    print("\nError metrics for the best performing XGBoost model (test set):")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test):.4f}")
    print(f"F1:        {f1_score(y_test, y_pred_test):.4f}")
    print(f"AUC:       {roc_auc_score(y_test, y_prob_test):.4f}")

    # Save the best model
    joblib.dump(best_model, Path("results") / "xgb_model.joblib")
    print("\nSaved best model to results/xgb_model.joblib")

    # Optional: save GridSearchCV summary
    results_df = pd.DataFrame(grid.cv_results_)
    results_df = results_df[["params", "mean_test_score", "rank_test_score"]]
    results_df.to_csv("output/xgboost_grid_summary.csv", index=False)
    print("Saved GridSearchCV summary to output/xgboost_grid_summary.csv")


# -----------------------------
if __name__ == "__main__":
    main()

Numeric columns (8): ['Age', 'TSH_Level', 'T3_Level', 'T4_Level', 'Nodule_Size', 'Risk_Factor_Score', 'TSH_Nodule_Interact', 'T3_TSH_Ratio']
Categorical columns (9): ['Gender', 'Country', 'Ethnicity', 'Family_History', 'Radiation_Exposure', 'Iodine_Deficiency', 'Smoking', 'Obesity', 'Diabetes']
Fitting 5 folds for each of 2 candidates, totalling 10 fits


/var/folders/0y/ttj5hqx15yj0v5c0f5db72lr0000gp/T/ipykernel_26491/105580613.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()



Best performing XGBoost model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'TSH_Level',
                                                   'T3_Level', 'T4_Level',
                                                   'Nodule_Size',
                                                   'Risk_Factor_Score',
                                                   'TSH_Nodule_Interact',
                                                   'T3_TSH_Ratio']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                              